# Fine-tune PaddleOCR tiếng Việt - Kaggle v3

**KAGGLE-V3-NESTED-RECURSIVE**

Pipeline: tải dữ liệu, giải nén outer zip + nested zip nhiều tầng, kiểm tra, so sánh 5 pretrained trên cùng validation protocol, chọn baseline, fine-tune, test và phân tích lỗi.


## Cài đặt

In [ ]:
!python -m pip uninstall -y -q paddlepaddle paddlepaddle-gpu
!python -m pip install -q gdown pyyaml rapidfuzz
!python -m pip install -q paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip show paddlepaddle-gpu
!nvidia-smi


In [ ]:
from pathlib import Path
import copy
import json
import hashlib
import math
import os
import pickle
import random
import shutil
import subprocess
import sys
import time
import unicodedata
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import yaml
from rapidfuzz.distance import Levenshtein
from IPython.display import display

SEED = 2026
ABLATION_EPOCHS = 1
FINAL_EPOCHS = 4
BATCH_SIZE = 16
IGNORE_SPACE = True

DRIVE_ID = "1_NKW1CL49NKtnT92ddaNZwGcaFJOkUgM"
PADDLEOCR_REF = "v3.3.0"

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
WORK = ROOT / "paddleocr_vi"
PADDLE_DIR = WORK / "PaddleOCR"
DATA_EXTRACT = WORK / "data"
CONFIG_DIR = WORK / "configs"
OUTPUT_DIR = WORK / "output"
RESULTS_DIR = WORK / "results"
WEIGHT_DIR = WORK / "weights"
LOG_DIR = WORK / "logs"

for p in [WORK, DATA_EXTRACT, CONFIG_DIR, OUTPUT_DIR, RESULTS_DIR, WEIGHT_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)


In [ ]:
if not PADDLE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", PADDLEOCR_REF,
                    "https://github.com/PaddlePaddle/PaddleOCR.git", str(PADDLE_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PADDLE_DIR), "fetch", "--tags", "--force"], check=True)
    subprocess.run(["git", "-C", str(PADDLE_DIR), "checkout", PADDLEOCR_REF], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PADDLE_DIR / "requirements.txt")], check=True)

import paddle
print("Paddle:", paddle.__version__)
print("CUDA:", paddle.device.is_compiled_with_cuda())
print("PaddleOCR:", subprocess.check_output(["git", "-C", str(PADDLE_DIR), "describe", "--tags", "--always"], text=True).strip())
print("Seed:", SEED)


## Dữ liệu

In [ ]:
import gdown

ZIP_PATH = WORK / "vi_rec_100k.zip"
if not ZIP_PATH.exists():
    gdown.download(id=DRIVE_ID, output=str(ZIP_PATH), quiet=False)


def marker_for(zip_path):
    zip_path = Path(zip_path)
    token = hashlib.sha1(str(zip_path.resolve()).encode("utf-8")).hexdigest()[:12]
    return zip_path.parent / f".{zip_path.stem}.{token}.extracted"


def unzip_once(zip_path, dest):
    zip_path = Path(zip_path)
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    marker = marker_for(zip_path)
    if marker.exists():
        return False
    print("Giải nén:", zip_path)
    with zipfile.ZipFile(zip_path, "r", allowZip64=True) as zf:
        zf.extractall(dest)
    marker.touch()
    return True


def valid_zip(path):
    return path.is_file() and "__MACOSX" not in path.parts and not path.name.startswith("._")


# Luôn xử lý zip ngoài. Marker bảo đảm rerun không giải nén lại 3.81 GB.
unzip_once(ZIP_PATH, DATA_EXTRACT)

# Zip có thể lồng nhiều tầng: quét lại sau mỗi vòng giải nén.
round_idx = 0
while True:
    round_idx += 1
    nested = [z for z in DATA_EXTRACT.rglob("*.zip") if valid_zip(z)]
    pending = [z for z in nested if not marker_for(z).exists()]
    if not pending:
        break
    print(f"Nested round {round_idx}: {len(pending)} zip")
    for nested_zip in pending:
        unzip_once(nested_zip, nested_zip.parent)


def find_one(root, name):
    matches = [m for m in root.rglob(name) if "__MACOSX" not in m.parts]
    if not matches:
        zips = [str(x.relative_to(root)) for x in root.rglob("*.zip") if valid_zip(x)][:30]
        txts = [str(x.relative_to(root)) for x in root.rglob("*.txt") if "__MACOSX" not in x.parts][:30]
        raise FileNotFoundError(f"Không tìm thấy {name}. ZIP đã thấy: {zips}. TXT đã thấy: {txts}")
    return matches[0]


TRAIN_FILE = find_one(DATA_EXTRACT, "rec_train.txt")
VAL_FILE = find_one(DATA_EXTRACT, "rec_val.txt")
TEST_FILE = find_one(DATA_EXTRACT, "rec_test.txt")
DICT_FILE = find_one(DATA_EXTRACT, "vi_dict.txt")

sample_line = next(x for x in TRAIN_FILE.read_text(encoding="utf-8").splitlines() if x.strip())
sample_rel = sample_line.split("\t", 1)[0]

# Label có thể nằm sâu trong DACK/data/... nên dò ngược các parent cho tới khi resolve được ảnh.
candidates = [TRAIN_FILE.parent, *TRAIN_FILE.parents, DATA_EXTRACT]
DATA_DIR = next((x for x in candidates if (x / sample_rel).exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(f"Không resolve được ảnh mẫu từ label: {sample_rel}")

print("Data:", DATA_DIR)
print("Train:", TRAIN_FILE)
print("Val:", VAL_FILE)
print("Test:", TEST_FILE)
print("Dict:", DICT_FILE)


## Kiểm tra dữ liệu

In [ ]:
def canonical_key(rel):
    p = Path(rel)
    if p.is_absolute():
        return os.path.normpath(os.path.relpath(p, DATA_DIR))
    return os.path.normpath(rel)


def read_labels(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n\r")
            if not line:
                continue
            rel, text = line.split("\t", 1)
            rows.append((canonical_key(rel), unicodedata.normalize("NFC", text)))
    return rows

train_rows = read_labels(TRAIN_FILE)
val_rows = read_labels(VAL_FILE)
test_rows = read_labels(TEST_FILE)

lengths = np.array([len(t) for _, t in train_rows])
MAX_TEXT_LENGTH = max(25, int(math.ceil(lengths.max() / 8) * 8))

stats = pd.DataFrame({
    "split": ["train", "val", "test"],
    "so_mau": [len(train_rows), len(val_rows), len(test_rows)]
})
display(stats)
print("Do dai p50/p90/p95/p99/max:", np.percentile(lengths, [50, 90, 95, 99]).tolist(), int(lengths.max()))
print("max_text_length:", MAX_TEXT_LENGTH)

with open(DICT_FILE, "r", encoding="utf-8") as f:
    dict_chars = {unicodedata.normalize("NFC", x.rstrip("\n\r")) for x in f if x.rstrip("\n\r")}

all_chars = set("".join(t for _, t in train_rows + val_rows + test_rows)) - {" "}
missing_chars = sorted(all_chars - dict_chars)
missing_images = [rel for rel, _ in train_rows + val_rows + test_rows if not (DATA_DIR / rel).exists()]

print("Ky tu ngoai vi_dict:", missing_chars[:50], "count=", len(missing_chars))
print("Anh khong tim thay:", len(missing_images))
if missing_chars:
    raise ValueError("vi_dict.txt khong phu ky tu trong nhan")
if missing_images:
    raise FileNotFoundError(missing_images[:10])

In [ ]:
MODEL_BASE = "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model"

PRETRAINED_MODELS = [
    {
        "name": "PP-OCRv5_mobile_rec",
        "config": PADDLE_DIR / "configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml",
    },
    {
        "name": "latin_PP-OCRv5_mobile_rec",
        "config": PADDLE_DIR / "configs/rec/PP-OCRv5/multi_language/latin_PP-OCRv5_mobile_rec.yml",
    },
    {
        "name": "en_PP-OCRv5_mobile_rec",
        "config": PADDLE_DIR / "configs/rec/PP-OCRv5/multi_language/en_PP-OCRv5_mobile_rec.yaml",
    },
    {
        "name": "eslav_PP-OCRv5_mobile_rec",
        "config": PADDLE_DIR / "configs/rec/PP-OCRv5/multi_language/eslav_PP-OCRv5_mobile_rec.yml",
    },
    {
        "name": "cyrillic_PP-OCRv5_mobile_rec",
        "config": PADDLE_DIR / "configs/rec/PP-OCRv5/multi_language/cyrillic_PP-OCRv5_mobile_rec.yaml",
    },
]


def download(url, path):
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path


for model in PRETRAINED_MODELS:
    model["weight"] = WEIGHT_DIR / f"{model['name']}_pretrained.pdparams"
    download(f"{MODEL_BASE}/{model['name']}_pretrained.pdparams", model["weight"])

pd.DataFrame([{k: str(v) for k, v in m.items()} for m in PRETRAINED_MODELS])


In [ ]:
def run_process(args, log_name):
    log_path = LOG_DIR / log_name
    start = time.time()
    with open(log_path, "w", encoding="utf-8") as log:
        p = subprocess.run(args, cwd=PADDLE_DIR, stdout=log, stderr=subprocess.STDOUT, text=True)
    if p.returncode != 0:
        tail = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()[-60:]
        print("\n".join(tail))
        raise RuntimeError("Command failed: " + " ".join(map(str, args)))
    return time.time() - start


def run_infer(config_path, label_file, result_txt, log_name, pretrained=None, checkpoint=None):
    args = [
        sys.executable, "tools/infer_rec.py", "-c", str(config_path), "-o",
        f"Global.infer_img={DATA_DIR}",
        f"Global.infer_list={label_file}",
        f"Global.save_res_path={result_txt}",
        "Global.distributed=False",
    ]
    if pretrained is not None:
        args.append(f"Global.pretrained_model={pretrained}")
    if checkpoint is not None:
        args.append(f"Global.checkpoints={checkpoint}")
    return run_process(args, log_name)


def read_prediction_file(path):
    preds = {}
    scores = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            full_path, pred, score = parts[0], parts[1], parts[-1]
            rel = os.path.normpath(os.path.relpath(full_path, DATA_DIR))
            preds[rel] = unicodedata.normalize("NFC", pred)
            try:
                scores[rel] = float(score)
            except ValueError:
                scores[rel] = None
    return preds, scores


def metric_text(text):
    text = unicodedata.normalize("NFC", text)
    return text.replace(" ", "") if IGNORE_SPACE else text


def evaluate_predictions(label_file, pred_txt, jsonl_path=None):
    gt_rows = read_labels(label_file)
    preds, scores = read_prediction_file(pred_txt)
    correct = 0
    distances = []
    records = []
    for rel, gt in gt_rows:
        pred = preds.get(rel, "")
        a, b = metric_text(pred), metric_text(gt)
        correct += int(a == b)
        distances.append(Levenshtein.normalized_distance(a, b))
        records.append({"image": rel, "gt": gt, "pred": pred})
    result = {
        "acc": correct / len(gt_rows),
        "norm_edit_dis": 1 - float(np.mean(distances)),
        "n": len(gt_rows),
        "missing_pred": sum(1 for rel, _ in gt_rows if rel not in preds),
    }
    if jsonl_path is not None:
        with open(jsonl_path, "w", encoding="utf-8") as f:
            for r in records:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return result, records


def make_pretrained_eval_config(model):
    with open(model["config"], "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    cfg["Global"]["distributed"] = False
    cfg["Global"]["pretrained_model"] = str(model["weight"])
    cfg["Global"]["checkpoints"] = None
    cfg["Global"]["d2s_train_image_shape"] = [3, 48, 320]
    for op in cfg["Eval"]["dataset"]["transforms"]:
        key = next(iter(op))
        if key == "RecResizeImg":
            op[key]["image_shape"] = [3, 48, 320]
    path = CONFIG_DIR / f"pretrained_{model['name']}.yml"
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
    return path


## Chọn pretrained

Năm model được chạy **không fine-tune** trên cùng `rec_val.txt`, cùng kích thước ảnh `48×320`, cùng hàm inference và cùng metric. Mỗi model giữ dictionary native tương ứng với pretrained head. Model tốt nhất trên validation được chọn làm baseline và initialization cho fine-tune.


In [ ]:
pretrained_results = []

for model in PRETRAINED_MODELS:
    print("Running:", model["name"])
    eval_config = make_pretrained_eval_config(model)
    pred_txt = RESULTS_DIR / f"pretrained_{model['name']}_val.txt"
    seconds = run_infer(
        eval_config,
        VAL_FILE,
        pred_txt,
        f"pretrained_{model['name']}_val.log",
        pretrained=model["weight"],
    )
    metric, _ = evaluate_predictions(VAL_FILE, pred_txt)
    pretrained_results.append({
        "name": model["name"],
        **metric,
        "val_inference_seconds": seconds,
        "config": str(model["config"]),
        "weight": str(model["weight"]),
    })

pretrained_df = pd.DataFrame(pretrained_results).sort_values(
    ["acc", "norm_edit_dis", "val_inference_seconds"],
    ascending=[False, False, True],
).reset_index(drop=True)
pretrained_df.to_csv(RESULTS_DIR / "pretrained_model_results.csv", index=False)
display(pretrained_df)

best_pretrained_row = pretrained_df.iloc[0]
best_pretrained = next(m for m in PRETRAINED_MODELS if m["name"] == best_pretrained_row["name"])
BEST_BASE_CONFIG = Path(best_pretrained["config"])
BEST_PRETRAINED_WEIGHT = Path(best_pretrained["weight"])

with open(RESULTS_DIR / "best_pretrained.json", "w", encoding="utf-8") as f:
    json.dump({
        "name": best_pretrained["name"],
        "val_acc": float(best_pretrained_row["acc"]),
        "val_norm_edit_dis": float(best_pretrained_row["norm_edit_dis"]),
    }, f, ensure_ascii=False, indent=2)

print("Best pretrained:", best_pretrained["name"])

BASELINE_CONFIG = make_pretrained_eval_config(best_pretrained)
BASELINE_TXT = RESULTS_DIR / "baseline_test.txt"
BASELINE_JSONL = RESULTS_DIR / "pred_test_baseline.jsonl"
baseline_time = run_infer(
    BASELINE_CONFIG,
    TEST_FILE,
    BASELINE_TXT,
    "baseline_test.log",
    pretrained=BEST_PRETRAINED_WEIGHT,
)
baseline_metric, baseline_records = evaluate_predictions(TEST_FILE, BASELINE_TXT, BASELINE_JSONL)
baseline_metric["inference_seconds"] = baseline_time
pd.DataFrame([baseline_metric])


## Fine-tune

Sau khi chọn pretrained tốt nhất, thử `width=640` và `width=960`. Các yếu tố khác giữ nguyên.


In [ ]:
CONFIGS = [
    {"name": "width_640", "width": 640, "recconaug": True},
    {"name": "width_960", "width": 960, "recconaug": True},
]


def make_config(spec, epochs, suffix="ablation"):
    with open(BEST_BASE_CONFIG, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    name = f"{spec['name']}_{suffix}"
    width = int(spec["width"])
    save_dir = OUTPUT_DIR / name

    g = cfg["Global"]
    g["model_name"] = name
    g["epoch_num"] = int(epochs)
    g["seed"] = SEED
    g["distributed"] = False
    g["save_model_dir"] = str(save_dir)
    g["save_epoch_step"] = 1
    g["eval_batch_step"] = [0, 1000000]
    g["pretrained_model"] = str(BEST_PRETRAINED_WEIGHT)
    g["checkpoints"] = None
    g["character_dict_path"] = str(DICT_FILE)
    g["max_text_length"] = MAX_TEXT_LENGTH
    g["use_space_char"] = True
    g["d2s_train_image_shape"] = [3, 48, width]

    cfg["Optimizer"]["lr"]["learning_rate"] = 0.0005
    cfg["Optimizer"]["lr"]["warmup_epoch"] = 0 if epochs <= 1 else 1
    cfg["Metric"]["ignore_space"] = IGNORE_SPACE

    for head in cfg["Architecture"]["Head"]["head_list"]:
        if "NRTRHead" in head:
            head["NRTRHead"]["max_text_length"] = MAX_TEXT_LENGTH

    train_ds = cfg["Train"]["dataset"]
    train_ds["data_dir"] = str(DATA_DIR)
    train_ds["label_file_list"] = [str(TRAIN_FILE)]

    new_transforms = []
    for op in train_ds["transforms"]:
        key = next(iter(op))
        if key == "RecConAug":
            op[key]["image_shape"] = [48, width, 3]
            op[key]["max_text_length"] = MAX_TEXT_LENGTH
        if key == "MultiLabelEncode":
            if op[key] is None:
                op[key] = {}
            op[key]["max_text_length"] = MAX_TEXT_LENGTH
        new_transforms.append(op)
    train_ds["transforms"] = new_transforms

    sampler = cfg["Train"]["sampler"]
    sampler["scales"] = [[width, 32], [width, 48], [width, 64]]
    sampler["first_bs"] = BATCH_SIZE
    sampler["fix_bs"] = True
    cfg["Train"]["loader"]["batch_size_per_card"] = BATCH_SIZE
    cfg["Train"]["loader"]["num_workers"] = 4

    eval_ds = cfg["Eval"]["dataset"]
    eval_ds["data_dir"] = str(DATA_DIR)
    eval_ds["label_file_list"] = [str(VAL_FILE)]
    for op in eval_ds["transforms"]:
        key = next(iter(op))
        if key == "RecResizeImg":
            op[key]["image_shape"] = [3, 48, width]
        if key == "MultiLabelEncode":
            if op[key] is None:
                op[key] = {}
            op[key]["max_text_length"] = MAX_TEXT_LENGTH
    cfg["Eval"]["loader"]["batch_size_per_card"] = BATCH_SIZE
    cfg["Eval"]["loader"]["num_workers"] = 4

    path = CONFIG_DIR / f"{name}.yml"
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
    return path, save_dir


ablation_config_paths = {}
for spec in CONFIGS:
    path, save_dir = make_config(spec, ABLATION_EPOCHS, "ablation")
    ablation_config_paths[spec["name"]] = (path, save_dir)

pd.DataFrame(CONFIGS)


In [ ]:
def checkpoint_epoch(prefix):
    states = Path(str(prefix) + ".states")
    if not states.exists():
        return 0
    with open(states, "rb") as f:
        data = pickle.load(f)
    return int(data.get("epoch", 0))


def train_with_resume(cfg_path, save_dir, epochs, log_name):
    latest = save_dir / "latest"
    time_file = save_dir / "train_seconds.txt"
    total_seconds = float(time_file.read_text().strip()) if time_file.exists() else 0.0
    done_epoch = checkpoint_epoch(latest)
    if done_epoch < epochs:
        args = [sys.executable, "tools/train.py", "-c", str(cfg_path)]
        if Path(str(latest) + ".pdparams").exists():
            args += ["-o", f"Global.checkpoints={latest}"]
        total_seconds += run_process(args, log_name)
        time_file.parent.mkdir(parents=True, exist_ok=True)
        time_file.write_text(str(total_seconds))
    return latest, total_seconds


ablation_results = []
for spec in CONFIGS:
    name = spec["name"]
    cfg_path, save_dir = ablation_config_paths[name]
    print("Training:", name)
    latest, train_seconds = train_with_resume(
        cfg_path, save_dir, ABLATION_EPOCHS, f"{name}_train.log"
    )
    pred_txt = RESULTS_DIR / f"{name}_val.txt"
    infer_seconds = run_infer(
        cfg_path, VAL_FILE, pred_txt, f"{name}_val.log", checkpoint=latest
    )
    metric, _ = evaluate_predictions(VAL_FILE, pred_txt)
    ablation_results.append({
        **spec,
        **metric,
        "train_seconds": train_seconds,
        "val_inference_seconds": infer_seconds,
        "config_path": str(cfg_path),
        "checkpoint": str(latest),
    })

ablation_df = pd.DataFrame(ablation_results).sort_values(
    ["acc", "norm_edit_dis"], ascending=[False, False]
).reset_index(drop=True)
ablation_df.to_csv(RESULTS_DIR / "ablation_width.csv", index=False)
display(ablation_df)


In [ ]:
best_row = ablation_df.iloc[0]
best_spec = next(x for x in CONFIGS if x["name"] == best_row["name"])

with open(RESULTS_DIR / "best_config.json", "w", encoding="utf-8") as f:
    json.dump(best_spec, f, ensure_ascii=False, indent=2)

print("Best width:", best_spec["width"])
print("Val acc:", best_row["acc"])
print("Val norm_edit_dis:", best_row["norm_edit_dis"])


### So sánh 640 và 960


In [ ]:
display(ablation_df[["name", "width", "acc", "norm_edit_dis", "train_seconds", "val_inference_seconds"]])


## Chọn tốt nhất


In [ ]:
print("Pretrained:", best_pretrained["name"])
print("Fine-tune width:", best_spec["width"])


## Fine-tune cuối

In [ ]:
FINAL_CONFIG, FINAL_SAVE_DIR = make_config(best_spec, FINAL_EPOCHS, "final")
FINAL_CKPT, final_train_seconds = train_with_resume(
    FINAL_CONFIG, FINAL_SAVE_DIR, FINAL_EPOCHS, "best_final_train.log"
)

FINAL_VAL_TXT = RESULTS_DIR / "best_final_val.txt"
run_infer(FINAL_CONFIG, VAL_FILE, FINAL_VAL_TXT, "best_final_val.log", checkpoint=FINAL_CKPT)
final_val_metric, _ = evaluate_predictions(VAL_FILE, FINAL_VAL_TXT)
print(final_val_metric)


## Test

In [ ]:
FINAL_TEST_TXT = RESULTS_DIR / "best_final_test.txt"
FINAL_TEST_JSONL = RESULTS_DIR / "pred_test_finetune.jsonl"

final_test_infer_seconds = run_infer(
    FINAL_CONFIG,
    TEST_FILE,
    FINAL_TEST_TXT,
    "best_final_test.log",
    checkpoint=FINAL_CKPT,
)
final_test_metric, final_test_records = evaluate_predictions(
    TEST_FILE,
    FINAL_TEST_TXT,
    FINAL_TEST_JSONL,
)

comparison = pd.DataFrame([
    {
        "model": f"Baseline: {best_pretrained['name']}",
        "acc": baseline_metric["acc"],
        "norm_edit_dis": baseline_metric["norm_edit_dis"],
        "train_seconds": 0.0,
    },
    {
        "model": "Fine-tune cua nhom",
        "acc": final_test_metric["acc"],
        "norm_edit_dis": final_test_metric["norm_edit_dis"],
        "train_seconds": final_train_seconds,
    },
])
comparison.to_csv(RESULTS_DIR / "comparison_test.csv", index=False)
display(comparison)


## Phân tích lỗi

Phân loại tự động chỉ là gợi ý; kiểm tra thủ công 20 dòng trước khi đưa vào báo cáo.

In [ ]:
TONE_MARKS = {"\u0300", "\u0301", "\u0303", "\u0309", "\u0323"}


def strip_tone(text):
    d = unicodedata.normalize("NFD", text)
    d = "".join(ch for ch in d if ch not in TONE_MARKS)
    return unicodedata.normalize("NFC", d)


def is_repeat_error(gt, pred):
    if len(pred) <= len(gt):
        return False
    for i in range(len(pred)):
        candidate = pred[:i] + pred[i + 1:]
        near_same = (i > 0 and pred[i] == pred[i - 1]) or (i + 1 < len(pred) and pred[i] == pred[i + 1])
        if candidate == gt and near_same:
            return True
    return False


def classify_error(gt, pred):
    if strip_tone(gt) == strip_tone(pred) and gt != pred:
        return "sai dau thanh"
    if is_repeat_error(gt, pred):
        return "lap ky tu"
    return "sai chu cai"

wrong = [r for r in final_test_records if metric_text(r["gt"]) != metric_text(r["pred"])]
rng = random.Random(SEED)
sample20 = rng.sample(wrong, min(20, len(wrong)))

for r in sample20:
    r["loai_loi_goi_y"] = classify_error(r["gt"], r["pred"])

error_df = pd.DataFrame(sample20)
error_df.to_csv(RESULTS_DIR / "error_analysis_20.csv", index=False)
display(error_df)

error_pct = (
    error_df["loai_loi_goi_y"]
    .value_counts(normalize=True)
    .mul(100)
    .rename_axis("loai_loi")
    .reset_index(name="phan_tram")
)
display(error_pct)

## Kết quả

In [ ]:
summary = {
    "seed": SEED,
    "paddle_version": paddle.__version__,
    "paddleocr_ref": PADDLEOCR_REF,
    "max_text_length": MAX_TEXT_LENGTH,
    "ablation_epochs": ABLATION_EPOCHS,
    "final_epochs": FINAL_EPOCHS,
    "batch_size": BATCH_SIZE,
    "best_pretrained": best_pretrained["name"],
    "best_config": best_spec,
    "baseline": baseline_metric,
    "final_val": final_val_metric,
    "final_test": final_test_metric,
    "final_train_seconds": final_train_seconds,
    "final_test_inference_seconds": final_test_infer_seconds,
}
with open(RESULTS_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

bundle = shutil.make_archive(str(WORK / "paddleocr_vi_results"), "zip", root_dir=RESULTS_DIR)
print("Results:", RESULTS_DIR)
print("Bundle:", bundle)
print("Final checkpoint:", FINAL_CKPT)
